# Colab + Google Drive Learning Cycle Template

This notebook provides a minimal workflow to:

1. Mount Google Drive
2. Prepare repository and dependencies on Colab
3. **Save acquired data (取得データ) to Google Drive** (`DATA_DIR` = Drive 上の `data/raw`)
4. Run training and prediction using Drive data
5. Sync learning/prediction artifacts (optuna_models, outputs, etc.) to Drive
6. Confirm output artifacts


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

REPO_DIR = '/content/kyotei_Prediction'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/takajin0114/kyotei_Prediction.git {REPO_DIR}

%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
import os

# Google Drive 上のプロジェクトルート（取得データ・成果物の保存先）
DRIVE_ROOT = '/content/drive/MyDrive/kyotei_prediction'
# 取得データ（raw）の保存先 → 必ず Google Drive 内
DATA_DIR = f'{DRIVE_ROOT}/data/raw'
YEAR_MONTH = '2026-02'
PREDICT_DATE = '2026-02-14'
N_TRIALS = 1
MINIMAL = True

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('DATA_DIR   =', DATA_DIR, '（取得データの保存先・Google Drive）')


In [ ]:
# 取得データを Google Drive に保存する（RUN_FETCH=True で実行）
# 保存先: DATA_DIR = /content/drive/MyDrive/kyotei_prediction/data/raw

RUN_FETCH = False

if RUN_FETCH:
    !python -m kyotei_predictor.tools.batch.batch_fetch_all_venues \
      --start-date 2026-02-01 \
      --end-date 2026-02-14 \
      --stadiums ALL \
      --output-data-dir "{DATA_DIR}"
    print('取得データの保存先（Google Drive）:', DATA_DIR)
    if os.path.exists(DATA_DIR):
        n = len([f for r, d, files in os.walk(DATA_DIR) for f in files])
        print(f'  → 保存ファイル数: {n}')


In [ ]:
import subprocess

cmd = [
    'python',
    'scripts/run_colab_learning_cycle.py',
    '--drive-root', DRIVE_ROOT,
    '--data-dir', DATA_DIR,
    '--year-month', YEAR_MONTH,
    '--n-trials', str(N_TRIALS),
    '--predict-date', PREDICT_DATE,
]
if MINIMAL:
    cmd.append('--minimal')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import os

# 予測結果はローカル outputs/ に出力後、Drive へ同期済み
pred_path = f'outputs/predictions_{PREDICT_DATE}.json'
drive_pred_path = f'{DRIVE_ROOT}/outputs/predictions_{PREDICT_DATE}.json'
print('prediction file (local):', os.path.exists(pred_path), pred_path)
print('prediction file (Drive):', os.path.exists(drive_pred_path), drive_pred_path)

path = pred_path if os.path.exists(pred_path) else (drive_pred_path if os.path.exists(drive_pred_path) else None)
if path and os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        pred = json.load(f)
    print('prediction_date:', pred.get('prediction_date'))
    print('summary:', pred.get('execution_summary'))
